# 🐝 Lab 3.2 — Inference Engineering: KV Cache, Sampling, Speculation & Quantization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab08_inference_engineering.ipynb)

**⏱ Time:** ~2 hours  ·  **Day 3 — Transformers & Beyond**

Training a model happens once; *serving* it happens billions of times. Buzz has a trained GPT and a hive full of impatient users. Today we make it **faster, cheaper, and better-behaved** at generation time, without retraining a single weight.

### 🎯 Learning objectives
- Implement a **KV cache**, prove it gives identical outputs, and benchmark tokens/sec with vs without it.
- Implement **greedy, temperature, top-k, top-p (nucleus) and min-p** sampling from scratch and explain how each truncates the distribution.
- Derive and implement the **speculative decoding** accept/reject rule, and verify empirically that it samples *exactly* from the target model.
- Quantize weights to **int8 and int4** (per-tensor, per-channel, group-wise absmax), and measure reconstruction error and perplexity impact.
- Explain **FlashAttention** (IO-awareness, online softmax) and **PagedAttention** (KV-cache memory management) at the level of "why it's fast".

### 🧰 Prerequisites
Lab 3.1 (the Llama-style GPT). Basic probability (sampling from a categorical, total-variation distance).

### 🌍 Why this matters in 2026
For deployed LLMs, inference is where most of the money goes. Serving stacks such as vLLM, SGLang, TensorRT-LLM and llama.cpp are built from combinations of the ideas in this notebook: a KV cache managed in pages, quantized weights (int8/int4/fp8 and friends), speculative decoding with a draft model or extra heads, fused attention kernels, and a sampler with temperature/top-p/min-p knobs. Knowing how each one works tells you which knob to turn when latency, memory or quality goes wrong.

## 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| 0 · Setup | Imports, seeds, device | 5 |
| 1 · Models to serve | Train a small target GPT and a tiny draft (~2 min CPU) | 10 |
| 2 · KV cache | Implement it, prove it's exact, benchmark tokens/sec | 20 |
| 3 · Sampling | Greedy, temperature, top-k, top-p, min-p, with truncation plots | 25 |
| 4 · Speculative decoding | Derive and verify the accept/reject rule; draft + target in action | 25 |
| 5 · Quantization | int8/int4, per-tensor vs per-channel vs group-wise; perplexity impact | 25 |
| 6 · FlashAttention & PagedAttention | Online softmax demo; KV-cache memory simulation | 10 |
| Wrap-up | Takeaways, interview corner, boss level | 5 |

## 0 · Setup 🛠️

In [ ]:
import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (normal users: leave unset)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=1337):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything()

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA",
                     "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})

from dataclasses import dataclass
import copy, warnings
# No %pip installs needed: everything used here ships with Colab.

print(f"torch {torch.__version__} | device = {DEVICE} | FAST = {FAST}")

## Part 1 · A small GPT to serve (target + draft) 🏭

This notebook is self-contained: below is a compact version of the Llama-style GPT from Lab 3.1 (RMSNorm, SwiGLU, RoPE, pre-norm, tied embeddings), with one addition: every attention layer accepts an optional **`cache`** and a **`start_pos`**, so it can process just the *new* tokens. You'll implement the cache itself in Part 2.

We train two models on Tiny Shakespeare:
* a **target** model (4 layers, d=128), the one we actually want samples from;
* a much smaller **draft** model (2 layers, d=64) for speculative decoding in Part 4.

Training takes ~3 minutes on CPU (both models), so start it and read ahead. ☕

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_DIR = os.environ.get("LAB_DATA_DIR", os.path.join(os.path.expanduser("~"), ".cache", "gradient_hive"))
os.makedirs(DATA_DIR, exist_ok=True)
FALLBACK = ("First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\n"
            "First Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\n"
            "First Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\n")

def load_tiny_shakespeare():
    path = os.path.join(DATA_DIR, "tinyshakespeare.txt")
    if os.path.exists(path):
        return open(path, encoding="utf-8").read()
    try:
        import urllib.request
        with urllib.request.urlopen(DATA_URL, timeout=30) as r:
            text = r.read().decode("utf-8")
        open(path, "w", encoding="utf-8").write(text)
        return text
    except Exception as e:
        print("⚠️  Download failed, using a tiny embedded fallback:", repr(e))
        return FALLBACK * 300

text = load_tiny_shakespeare()
chars = sorted(set(text)); VOCAB = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[int(i)] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
print(f"{len(text):,} chars, vocab {VOCAB}")

def get_batch(split, B, T):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - T - 1, (B,))
    x = torch.stack([d[i:i + T] for i in ix]); y = torch.stack([d[i + 1:i + T + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

In [ ]:
@dataclass
class Cfg:
    vocab_size: int = VOCAB
    block_size: int = 256
    n_layer: int = 4
    n_head: int = 4
    d_model: int = 128

def rope_angles(T, dh, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, dh, 2).float() / dh))
    ang = torch.outer(torch.arange(T).float(), inv)
    return ang.cos(), ang.sin()

def apply_rope(x, cos, sin):                # x: (B, H, t, dh); cos/sin: (t, dh/2), already sliced to positions
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2).type_as(x)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d):
        super().__init__(); h = 32 * math.ceil(int(8 * d / 3) / 32)
        self.w_gate = nn.Linear(d, h, bias=False); self.w_up = nn.Linear(d, h, bias=False); self.w_down = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.H, self.dh = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, cache=None, layer=0, start_pos=0):
        B, t, C = x.shape
        q, k, v = self.qkv(x).view(B, t, 3, self.H, self.dh).permute(2, 0, 3, 1, 4)   # (B, H, t, dh)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(layer, k, v, start_pos)          # now (B, H, start_pos + t, dh)
        S = k.shape[-2]
        if S == t:                         # no past: ordinary causal attention
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        elif t == 1:                       # one new token: it may see everything in the cache
            y = F.scaled_dot_product_attention(q, k, v)
        else:                              # a chunk of t new tokens on top of S - t cached ones
            mask = torch.ones(t, S, dtype=torch.bool, device=x.device).tril(diagonal=S - t)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.proj(y.transpose(1, 2).reshape(B, t, C))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1, self.attn, self.n2, self.mlp = RMSNorm(cfg.d_model), Attention(cfg), RMSNorm(cfg.d_model), SwiGLU(cfg.d_model)
    def forward(self, x, cos, sin, cache, layer, start_pos):
        x = x + self.attn(self.n1(x), cos, sin, cache, layer, start_pos)
        return x + self.mlp(self.n2(x))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.emb.weight
        cos, sin = rope_angles(cfg.block_size, cfg.d_model // cfg.n_head)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
        for name, p in self.named_parameters():
            if p.dim() == 2:
                std = 0.02 / math.sqrt(2 * cfg.n_layer) if name.endswith(("proj.weight", "w_down.weight")) else 0.02
                nn.init.normal_(p, 0.0, std)

    def forward(self, idx, targets=None, cache=None, start_pos=0):
        B, t = idx.shape
        assert start_pos + t <= self.cfg.block_size
        cos, sin = self.cos[start_pos:start_pos + t], self.sin[start_pos:start_pos + t]
        x = self.emb(idx)
        for i, blk in enumerate(self.blocks):
            x = blk(x, cos, sin, cache, i, start_pos)
        logits = self.lm_head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
def train_model(cfg, iters, B=16, lr=3e-3, label="model"):
    seed_everything()
    m = GPT(cfg).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    warm = max(1, iters // 15)
    t0 = time.time()
    for it in range(iters):
        cur = lr * (it + 1) / warm if it < warm else 0.1 * lr + 0.45 * lr * (1 + math.cos(math.pi * (it - warm) / max(1, iters - warm)))
        for g in opt.param_groups: g["lr"] = cur
        x, y = get_batch("train", B, cfg.block_size)
        _, loss = m(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        if it % max(1, iters // 5) == 0 or it == iters - 1:
            print(f"  [{label}] it {it:4d} loss {loss.item():.3f}  ({time.time() - t0:.0f}s)")
    return m.eval()

@torch.no_grad()
def eval_loss(m, n_batches=None, B=16):
    """Deterministic val loss on fixed, non-overlapping windows (so comparisons are apples-to-apples)."""
    n_batches = n_batches or (2 if FAST else 20)
    T = m.cfg.block_size
    losses = []
    for b in range(n_batches):
        starts = [(b * B + j) * T for j in range(B)]
        x = torch.stack([val_data[s:s + T] for s in starts]).to(DEVICE)
        y = torch.stack([val_data[s + 1:s + T + 1] for s in starts]).to(DEVICE)
        losses.append(m(x, y)[1].item())
    return float(np.mean(losses))

TARGET_ITERS, DRAFT_ITERS = (30, 20) if FAST else (800, 500)
target_cfg = Cfg(n_layer=4, n_head=4, d_model=128)
draft_cfg = Cfg(n_layer=2, n_head=2, d_model=64)
target = train_model(target_cfg, TARGET_ITERS, label="target")
draft = train_model(draft_cfg, DRAFT_ITERS, label="draft")
n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f"target: {n_params(target)/1e6:.2f}M params, val loss {eval_loss(target):.3f}")
print(f"draft : {n_params(draft)/1e6:.2f}M params, val loss {eval_loss(draft):.3f}")

> **🍯 Key takeaway:** A trained model is just a function from tokens to next-token logits. Everything in this lab changes *how we call* that function, not what it learned.

## Part 2 · The KV cache 🗄️

**The waste.** Naive generation calls `model(idx)` on the *whole* prefix at every step. But with a causal mask, the keys and values of tokens $1..t$ **never change** when token $t+1$ arrives. Recomputing them makes step $t$ cost $O(t)$ projections and the whole generation $O(n^2)$ projections.

**The fix.** Store each layer's $K$ and $V$ as you go. At step $t$ only the newest token is processed: compute its $q_t, k_t, v_t$, append $k_t, v_t$ to the cache, and attend $q_t$ against all cached keys:
$$o_t = \operatorname{softmax}\!\Big(\tfrac{q_t K_{1:t}^\top}{\sqrt d}\Big) V_{1:t}.$$

Generation now has two phases:
* **prefill**: process the prompt in one parallel pass, filling the cache (compute-bound, like training);
* **decode**: one token at a time, reading the whole cache each step (memory-bandwidth-bound).

**The price: memory.** Per token, the cache holds $2 \times n_{layers} \times n_{kv\,heads} \times d_{head}$ numbers. For a Llama-3-8B-like config (32 layers, 8 KV heads via GQA, $d_{head}=128$, fp16) that is $2\cdot 32\cdot 8\cdot 128\cdot 2\text{ B} = 128\text{ KiB per token}$, so **1 GiB per 8k-token sequence**. That's why GQA/MQA, multi-head *latent* attention (DeepSeek-V2/V3), KV quantization and paged memory (Part 6) exist.

In [ ]:
def kv_cache_bytes(n_layers, n_kv_heads, d_head, seq_len, batch=1, bytes_per=2):
    return 2 * n_layers * n_kv_heads * d_head * seq_len * batch * bytes_per

for name, (L, kvh, dh) in {"Llama-3-8B-like (GQA, 8 KV heads)": (32, 8, 128),
                           "same but full MHA (32 KV heads)": (32, 32, 128),
                           "our target model": (4, 4, 32)}.items():
    print(f"{name:36s}: {kv_cache_bytes(L, kvh, dh, 1) / 1024:8.1f} KiB/token | "
          f"{kv_cache_bytes(L, kvh, dh, 8192) / 2**30:6.2f} GiB per 8k-token sequence")

### 🧪 Your Turn #1 — implement `KVCache.update`
We pre-allocate `k`, `v` buffers of shape `(n_layer, B, H, max_len, d_head)`. `update(layer, k_new, v_new, start_pos)` must:
1. write `k_new`, `v_new` (shape `(B, H, t, d_head)`) into positions `start_pos : start_pos + t` of that layer's buffers;
2. return the **filled prefix** `[:start_pos + t]` of the layer's K and V.

Because writes are indexed by `start_pos`, "rolling back" the cache (which speculative decoding needs) is free: just call again with a smaller `start_pos`; stale entries beyond it are overwritten or ignored.

In [ ]:
class KVCache:
    def __init__(self, cfg, batch=1, max_len=None, device=DEVICE, dtype=torch.float32):
        max_len = max_len or cfg.block_size
        shape = (cfg.n_layer, batch, cfg.n_head, max_len, cfg.d_model // cfg.n_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)

    def update(self, layer, k_new, v_new, start_pos):
        # TODO 1: write k_new / v_new into self.k[layer] / self.v[layer] at positions start_pos : start_pos + t
        # TODO 2: return the filled prefixes (positions 0 : start_pos + t)
        raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
class KVCache:
    def __init__(self, cfg, batch=1, max_len=None, device=DEVICE, dtype=torch.float32):
        max_len = max_len or cfg.block_size
        shape = (cfg.n_layer, batch, cfg.n_head, max_len, cfg.d_model // cfg.n_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)

    def update(self, layer, k_new, v_new, start_pos):
        t = k_new.shape[-2]
        self.k[layer, :, :, start_pos:start_pos + t] = k_new
        self.v[layer, :, :, start_pos:start_pos + t] = v_new
        return self.k[layer, :, :, :start_pos + t], self.v[layer, :, :, :start_pos + t]

In [ ]:
@torch.no_grad()
def generate_nocache(model, prompt_ids, n_new, pick):
    idx = torch.tensor([prompt_ids], device=DEVICE)
    for _ in range(n_new):
        logits, _ = model(idx)                       # recompute EVERYTHING each step
        idx = torch.cat([idx, pick(logits[:, -1]).view(1, 1)], 1)
    return idx[0].tolist()

@torch.no_grad()
def generate_cached(model, prompt_ids, n_new, pick):
    cache = KVCache(model.cfg, batch=1)
    idx = torch.tensor([prompt_ids], device=DEVICE)
    logits, _ = model(idx, cache=cache, start_pos=0)   # prefill
    out, pos = list(prompt_ids), len(prompt_ids)
    for i in range(n_new):
        nxt = pick(logits[:, -1]).view(1, 1)
        out.append(nxt.item())
        if i == n_new - 1:
            break
        logits, _ = model(nxt, cache=cache, start_pos=pos)  # decode: ONE token in, one token out
        pos += 1
    return out

greedy_pick = lambda logits: logits.argmax(-1)

### ✔️ Checkpoint — the cache must change speed, never answers

In [ ]:
prompt = encode("ROMEO:\n")
a = generate_nocache(target, prompt, 60, greedy_pick)
b = generate_cached(target, prompt, 60, greedy_pick)
assert a == b, "cached and uncached greedy generations differ!"

# logits check: incremental (prefill 10, then 1 token at a time) vs one full forward pass
x = val_data[:40].unsqueeze(0).to(DEVICE)
with torch.no_grad():
    full, _ = target(x)
    cache = KVCache(target.cfg)
    pieces = [target(x[:, :10], cache=cache, start_pos=0)[0]]
    pieces += [target(x[:, p:p + 1], cache=cache, start_pos=p)[0] for p in range(10, 30)]
    pieces += [target(x[:, 30:40], cache=cache, start_pos=30)[0]]      # a multi-token chunk on top of a cache
inc = torch.cat(pieces, 1)
assert torch.allclose(full, inc, atol=1e-4), (full - inc).abs().max()
print("✔️ KV cache is exact. Greedy sample:\n" + decode(b))

### 🔮 Predict-then-run
Our model has a 256-token context. Generating 250 tokens without a cache processes $1+2+\dots+250 \approx 31{,}000$ token-positions; with a cache, only ~250. Will the wall-clock speedup be ~125×? Guess, then run. (Hint: think about what dominates the cost of a forward pass on a *tiny* model.)

In [ ]:
lengths = [16, 32, 64] if FAST else [16, 32, 64, 128, 192, 250]
res = {"no cache": [], "KV cache": []}
prompt = encode("\n")
for n_new in lengths:
    for name, fn in [("no cache", generate_nocache), ("KV cache", generate_cached)]:
        fn(target, prompt, 4, greedy_pick)                     # warm-up
        t0 = time.perf_counter(); fn(target, prompt, n_new, greedy_pick); dt = time.perf_counter() - t0
        res[name].append(n_new / dt)
    print(f"n_new={n_new:4d} | no cache {res['no cache'][-1]:7.1f} tok/s | KV cache {res['KV cache'][-1]:7.1f} tok/s | "
          f"speedup x{res['KV cache'][-1] / res['no cache'][-1]:.1f}")

fig, ax = plt.subplots(figsize=(7, 3.8))
for name, mk in [("no cache", "o-"), ("KV cache", "s-")]:
    ax.plot(lengths, res[name], mk, label=name)
ax.set_xlabel("tokens generated"); ax.set_ylabel("throughput (tokens / s)"); ax.set_yscale("log")
ax.set_title(f"Generation throughput, batch 1, {DEVICE.upper()}"); ax.legend(); plt.show()

The speedup grows with length (the no-cache cost per token grows linearly with the prefix), but on a tiny CPU model it is far below the FLOP ratio: each forward pass is dominated by **fixed overhead** (Python, kernel launches, reading the weights from memory) rather than arithmetic. This is the central fact of LLM decoding: **at batch size 1, decoding is memory-bandwidth / overhead bound, not compute bound.** Each step must stream *all* weights (and the whole KV cache) through the chip to produce *one* token. Batching many users, speculative decoding (Part 4) and quantization (Part 5) are all ways to get more tokens out of each trip through memory.

> **🍯 Key takeaway:** The KV cache trades memory for compute: identical outputs, one new token per step, but memory that grows with context. At batch size 1, decoding is bound by memory traffic and overhead, not FLOPs.

## Part 3 · Sampling strategies 🎲

The model hands us logits $z \in \mathbb{R}^V$; *how* we turn them into a token is a separate design decision with a big effect on output quality.

| Strategy | Rule | Character |
|---|---|---|
| **Greedy** | $\arg\max_i z_i$ | deterministic; prone to loops ("the the the") |
| **Temperature** $\tau$ | $p_i \propto \exp(z_i/\tau)$ | $\tau\to0$ = greedy, $\tau=1$ = model, $\tau>1$ = flatter |
| **Top-k** | keep the $k$ largest $p_i$, renormalise | fixed-size shortlist, ignores how confident the model is |
| **Top-p / nucleus** (Holtzman et al., 2020) | keep the smallest set with $\sum p_i \ge p$ | shortlist adapts to the shape of the distribution |
| **Min-p** (Nguyen et al., 2024) | keep $i$ with $p_i \ge p_{min}\cdot \max_j p_j$ | threshold scales with the model's confidence; robust at high $\tau$ |

All the truncation methods work by setting the removed logits to $-\infty$ and re-normalising. We apply temperature first, then the filters (the default order in Hugging Face `generate`; libraries differ, and the order matters at high temperature).

In [ ]:
def apply_top_k(logits, k):
    kth = torch.topk(logits, k, dim=-1).values[..., -1:]
    return logits.masked_fill(logits < kth, float("-inf"))

### 🧪 Your Turn #2 — top-p (nucleus) filtering
1. Sort logits descending; compute `probs = softmax(sorted)` and `cum = probs.cumsum(-1)`.
2. Remove a token if the mass **before** it already reaches `p`: `remove = (cum - probs) >= p`. (The top token is never removed.)
3. Scatter the removal mask back to the original order (`torch.zeros_like(...).scatter(-1, sorted_idx, remove)`) and fill with `-inf`.

### 🧪 Your Turn #3 — min-p filtering
Keep token $i$ iff $p_i \ge p_{min} \cdot \max_j p_j$. One line with `softmax`, `amax(..., keepdim=True)` and `masked_fill`.

In [ ]:
def apply_top_p(logits, p):
    # TODO
    raise NotImplementedError

def apply_min_p(logits, min_p):
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def apply_top_p(logits, p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
    probs = sorted_logits.softmax(-1)
    cum = probs.cumsum(-1)
    remove_sorted = (cum - probs) >= p
    remove = torch.zeros_like(remove_sorted).scatter(-1, sorted_idx, remove_sorted)
    return logits.masked_fill(remove, float("-inf"))

def apply_min_p(logits, min_p):
    probs = logits.softmax(-1)
    return logits.masked_fill(probs < min_p * probs.amax(-1, keepdim=True), float("-inf"))

In [ ]:
def sample(logits, temperature=1.0, top_k=None, top_p=None, min_p=None, generator=None):
    """logits: (B, V). Returns (B,) token ids."""
    if temperature == 0:
        return logits.argmax(-1)
    logits = logits.float() / temperature
    if top_k is not None: logits = apply_top_k(logits, top_k)
    if top_p is not None: logits = apply_top_p(logits, top_p)
    if min_p is not None: logits = apply_min_p(logits, min_p)
    probs = logits.softmax(-1)
    return torch.multinomial(probs.cpu(), 1, generator=generator).view(-1).to(logits.device)

# ✔️ Checkpoint
z = torch.tensor([[0.50, 0.30, 0.10, 0.05, 0.05]]).log()
kept = lambda l: torch.isfinite(l)[0].tolist()
assert kept(apply_top_p(z, 0.75)) == [True, True, False, False, False]
assert kept(apply_top_p(z, 0.50)) == [True, False, False, False, False]
assert kept(apply_top_p(z, 0.99)) == [True, True, True, True, True]
assert kept(apply_min_p(z, 0.15)) == [True, True, True, False, False]     # threshold = 0.15 * 0.5 = 0.075
assert kept(apply_min_p(z, 0.7)) == [True, False, False, False, False]
assert kept(apply_top_k(z, 2)) == [True, True, False, False, False]
zb = torch.randn(7, 50)                                                      # batched + unsorted input
assert torch.equal(torch.isfinite(apply_top_p(zb, 0.9)).sum(-1) >= 1, torch.ones(7, dtype=torch.bool))
print("✔️ top-p and min-p behave")

### Visualising truncation on real next-character distributions
We take two contexts: one where the next character is nearly certain, one where it's genuinely open. Watch which strategies **adapt** their shortlist size.

In [ ]:
@torch.no_grad()
def next_logits(ctx):
    return target(torch.tensor([encode(ctx)], device=DEVICE))[0][0, -1].float().cpu()

contexts = {"confident": "First Citizen:\nBefore we proceed any further, hear me spe",
            "open-ended": "ROMEO:\nI will "}
strategies = {"top-k (k=8)": lambda l: apply_top_k(l, 8), "top-p (p=0.9)": lambda l: apply_top_p(l, 0.9),
              "min-p (0.1)": lambda l: apply_min_p(l, 0.1)}
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for r, (cname, ctx) in enumerate(contexts.items()):
    lg = next_logits(ctx); pr = lg.softmax(-1)
    order = pr.argsort(descending=True)[:25]
    for c, (sname, fn) in enumerate(strategies.items()):
        keep = torch.isfinite(fn(lg.unsqueeze(0))[0])
        ax = axes[r, c]
        ax.bar(range(25), pr[order].numpy(), color=[HIVE[3] if keep[i] else "#D9D4C7" for i in order])
        ax.set_xticks(range(25)); ax.set_xticklabels([repr(itos[int(i)])[1:-1] for i in order], fontsize=8)
        ax.set_title(f"{cname}: {sname} keeps {int(keep.sum())}")
        if c == 0: ax.set_ylabel("p(next char)")
plt.suptitle("teal = kept, grey = truncated   (context ends: '..." + contexts["confident"][-12:].replace(chr(10), "⏎") +
             "'  /  '..." + contexts["open-ended"][-12:].replace(chr(10), "⏎") + "')")
plt.tight_layout(); plt.show()

In [ ]:
# Temperature reshapes the whole distribution; entropy rises monotonically with tau.
lg = next_logits(contexts["open-ended"])
taus = [0.3, 0.7, 1.0, 1.5, 2.5]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
order = lg.softmax(-1).argsort(descending=True)[:20]
for i, tau in enumerate(taus):
    axes[0].plot((lg / tau).softmax(-1)[order].numpy(), "o-", ms=3, label=f"τ={tau}")
axes[0].set_xticks(range(20)); axes[0].set_xticklabels([repr(itos[int(i)])[1:-1] for i in order])
axes[0].legend(); axes[0].set_title("next-char distribution vs temperature")
ts = np.linspace(0.1, 3, 60)
ent = [-(p * p.clamp_min(1e-12).log()).sum().item() for p in [(lg / t).softmax(-1) for t in ts]]
axes[1].plot(ts, ent); axes[1].axhline(math.log(VOCAB), ls=":", color="gray", label="ln V (uniform)")
axes[1].set_xlabel("temperature τ"); axes[1].set_ylabel("entropy (nats)"); axes[1].legend(); axes[1].set_title("entropy vs τ")
plt.tight_layout(); plt.show()

In [ ]:
# How many tokens survive truncation, across many real contexts?
with torch.no_grad():
    xs = val_data[:target.cfg.block_size * 8].view(8, -1).to(DEVICE)
    all_logits = target(xs)[0].reshape(-1, VOCAB).float().cpu()
counts = {s: torch.isfinite(fn(all_logits)).sum(-1).numpy() for s, fn in strategies.items()}
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.arange(0.5, 30.5, 1)
for s, c in counts.items():
    ax.hist(c, bins=bins, alpha=0.6, label=f"{s} (mean {c.mean():.1f})")
ax.set_xlabel("# tokens kept"); ax.set_ylabel("# contexts"); ax.legend()
ax.set_title("top-k is rigid; top-p and min-p adapt to the model's uncertainty"); plt.show()

In [ ]:
def gen_with(n_new=200, seed=0, **kw):
    g = torch.Generator().manual_seed(seed)
    return decode(generate_cached(target, encode("ROMEO:\n"), n_new, lambda l: sample(l, generator=g, **kw)))

def distinct_4grams(s):
    grams = [s[i:i + 4] for i in range(len(s) - 3)]
    return len(set(grams)) / max(1, len(grams))

settings = {"greedy": dict(temperature=0), "τ=1.0 (pure)": dict(temperature=1.0),
            "τ=1.6 (hot)": dict(temperature=1.6), "τ=1.6 + top-p 0.9": dict(temperature=1.6, top_p=0.9),
            "τ=1.6 + min-p 0.1": dict(temperature=1.6, min_p=0.1), "τ=0.8 + top-k 20": dict(temperature=0.8, top_k=20)}
n_show = 80 if FAST else 200
for name, kw in settings.items():
    s = gen_with(n_show, **kw)
    print(f"━━ {name}  (distinct 4-grams {distinct_4grams(s):.2f}) ━━\n{s[7:]}\n")

Typical pattern: greedy loops; pure high temperature produces garbled character soup; **top-p / min-p at the same high temperature stay coherent** because they chop off the long tail of junk characters *before* sampling, while keeping the diversity that the high temperature buys. That's the core argument of the min-p paper.

> **🍯 Key takeaway:** Temperature reshapes the distribution; top-k truncates rigidly; top-p and min-p truncate adaptively, which keeps hot sampling coherent.

## Part 4 · Speculative decoding 🏎️

**Idea** (Leviathan et al., 2023; Chen et al., 2023). Decoding is memory-bound: scoring **γ+1 tokens in one forward pass costs about the same** as scoring 1. So let a cheap **draft** model $q$ guess γ tokens, then run the expensive **target** $p$ *once* over all of them in parallel and keep the longest acceptable prefix.

**The accept/reject rule** that makes this *exact*: for a drafted token $x \sim q$,
$$\text{accept with probability } \min\!\Big(1, \frac{p(x)}{q(x)}\Big), \qquad \text{otherwise resample } x' \sim \frac{\max(0,\, p - q)}{\sum_y \max(0,\, p(y)-q(y))}.$$

**Why the output is distributed exactly as $p$.** The probability of emitting $x$ is
$$\underbrace{q(x)\min\!\Big(1,\tfrac{p(x)}{q(x)}\Big)}_{\text{accepted}} + \underbrace{\Big(1 - \textstyle\sum_y \min(p(y), q(y))\Big)}_{P(\text{reject})}\cdot\frac{\max(0, p(x)-q(x))}{\sum_y \max(0, p(y)-q(y))}.$$
The first term is $\min(p(x), q(x))$. Since $\sum_y \max(0, p-q) = 1 - \sum_y \min(p,q)$, the second term is just $\max(0, p(x)-q(x))$. And $\min(p,q) + \max(0, p-q) = p$. ∎

The acceptance rate per token is $\alpha = \sum_x \min(p(x), q(x)) = 1 - \mathrm{TV}(p, q)$, so a draft that *agrees* with the target gets more tokens per target call: with i.i.d. acceptance, expected tokens per target pass $= \frac{1-\alpha^{\gamma+1}}{1-\alpha}$.

### 🧪 Your Turn #4 — vectorised accept/reject on a toy distribution
Given target `p` and draft `q` (both 1-D probability vectors), draw `n` drafted tokens from `q`, accept each with prob `min(1, p[x]/q[x])`, and replace each rejected one with a sample from the normalised residual `max(0, p − q)`. Return `(samples, accepted_mask)`.

In [ ]:
def speculative_sample_toy(p, q, n, generator=None):
    # TODO 1: x = n draws from q          (torch.multinomial(q, n, replacement=True, generator=generator))
    # TODO 2: accept = uniform < min(1, p[x] / q[x])
    # TODO 3: residual = clamp(p - q, min=0), normalised; resample the rejected positions from it
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def speculative_sample_toy(p, q, n, generator=None):
    x = torch.multinomial(q, n, replacement=True, generator=generator)
    accept = torch.rand(n, generator=generator) < (p[x] / q[x]).clamp(max=1.0)
    residual = (p - q).clamp(min=0)
    residual = residual / residual.sum()
    n_rej = int((~accept).sum())
    if n_rej > 0:
        x = x.clone()
        x[~accept] = torch.multinomial(residual, n_rej, replacement=True, generator=generator)
    return x, accept

In [ ]:
# ✔️ Checkpoint — empirical distribution must match p, NOT q
g = torch.Generator().manual_seed(0)
V_toy = 10
p_toy = torch.softmax(torch.randn(V_toy, generator=g) * 1.5, 0)
q_toy = torch.softmax(torch.randn(V_toy, generator=g) * 1.5, 0)
N = 20_000 if FAST else 400_000
xs, acc = speculative_sample_toy(p_toy, q_toy, N, generator=g)
emp = torch.bincount(xs, minlength=V_toy).float() / N
tv = 0.5 * (emp - p_toy).abs().sum().item()
alpha_theory = torch.minimum(p_toy, q_toy).sum().item()
print(f"TV(empirical, p) = {tv:.4f}   |   TV(q, p) = {0.5 * (q_toy - p_toy).abs().sum():.3f}")
print(f"acceptance rate: empirical {acc.float().mean():.4f} vs theory Σmin(p,q) = {alpha_theory:.4f}")
assert tv < (0.02 if FAST else 0.005)
assert abs(acc.float().mean().item() - alpha_theory) < (0.02 if FAST else 0.005)

w = 0.27
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(np.arange(V_toy) - w, q_toy, w, label="draft q", color=HIVE[4])
ax.bar(np.arange(V_toy), p_toy, w, label="target p", color=HIVE[0])
ax.bar(np.arange(V_toy) + w, emp, w, label="speculative samples", color=HIVE[3])
ax.set_xlabel("token"); ax.legend(); ax.set_title("speculative sampling reproduces the target distribution exactly")
plt.show()
print("✔️ accept/reject rule is distribution-preserving 🐝")

### The real thing: draft + target with KV caches
Bookkeeping is the hard part. We keep, for each model, the number of **valid** cache entries. Each round:
1. the draft proposes γ tokens autoregressively (feeding only tokens its cache hasn't seen);
2. the target scores *all* γ drafted positions (+1 bonus position) in **one** forward pass;
3. walk left-to-right applying the accept/reject rule; at the first rejection emit a residual sample and stop; if everything is accepted, emit a free bonus token from the target's last distribution;
4. "roll back" both caches by shrinking their valid lengths (the KV cache's `start_pos` indexing makes this free).

In [ ]:
@torch.no_grad()
def speculative_generate(target, draft, prompt_ids, n_new, gamma=4, temperature=1.0, seed=0, trace=None):
    g = torch.Generator().manual_seed(seed)
    seq = list(prompt_ids)
    max_len = len(seq) + n_new + gamma + 2
    assert max_len <= target.cfg.block_size
    tcache, dcache = KVCache(target.cfg, max_len=max_len), KVCache(draft.cfg, max_len=max_len)
    t_valid = d_valid = 0
    stats = {"target_calls": 0, "draft_calls": 0, "proposed": 0, "evaluated": 0, "accepted": 0}
    to_probs = lambda logits: (logits.float() / temperature).softmax(-1).cpu()

    while len(seq) - len(prompt_ids) < n_new:
        # 1) draft proposes gamma tokens
        cur, drafts, qs = list(seq), [], []
        for _ in range(gamma):
            inp = torch.tensor([cur[d_valid:]], device=DEVICE)
            q = to_probs(draft(inp, cache=dcache, start_pos=d_valid)[0][0, -1])
            d_valid = len(cur); stats["draft_calls"] += 1
            x = torch.multinomial(q, 1, generator=g).item()
            drafts.append(x); qs.append(q); cur.append(x)
        # 2) target scores everything in ONE pass
        inp = torch.tensor([cur[t_valid:]], device=DEVICE)
        logits = target(inp, cache=tcache, start_pos=t_valid)[0][0]
        ps = to_probs(logits[len(seq) - 1 - t_valid:])          # gamma + 1 distributions
        stats["target_calls"] += 1
        # 3) accept / reject left to right
        n_acc = 0
        for i, x in enumerate(drafts):
            stats["evaluated"] += 1                     # drafts after a rejection are never tested
            if torch.rand(1, generator=g).item() < min(1.0, (ps[i][x] / qs[i][x]).item()):
                n_acc += 1
            else:
                resid = (ps[i] - qs[i]).clamp(min=0)
                new = torch.multinomial(resid / resid.sum() if resid.sum() > 0 else ps[i], 1, generator=g).item()
                break
        else:
            new = torch.multinomial(ps[gamma], 1, generator=g).item()   # all accepted: bonus token
        stats["proposed"] += gamma; stats["accepted"] += n_acc
        if trace is not None:   # for the animation below
            trace.append({"accepted": drafts[:n_acc], "new": new, "kind": "bonus" if n_acc == gamma else "resample"})
        seq = seq + drafts[:n_acc] + [new]
        # 4) roll back caches: entries for positions < len(seq) - 1 are valid
        t_valid = min(len(cur), len(seq) - 1)
        d_valid = min(d_valid, len(seq) - 1)
    return seq[:len(prompt_ids) + n_new], stats

@torch.no_grad()
def plain_generate(model, prompt_ids, n_new, temperature=1.0, seed=0):
    g = torch.Generator().manual_seed(seed)
    return generate_cached(model, prompt_ids, n_new, lambda l: sample(l, temperature=temperature, generator=g))

prompt = encode("ROMEO:\n")
out, st = speculative_generate(target, draft, prompt, 120 if not FAST else 40, gamma=4)
print(decode(out)); print(st, f"acceptance rate per tested draft {st['accepted'] / st['evaluated']:.2f}")

### 🎬 Speculation, round by round
Each frame is **one target forward pass**. Teal characters were proposed by the draft and accepted; coral ones were sampled by the target after it rejected a draft token; purple ones are the free "bonus" token you get when *all* drafts are accepted. Long teal runs = several tokens for the price of one big-model call.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

trace = []
_ = speculative_generate(target, draft, prompt, 40 if FAST else 150, gamma=4, seed=1, trace=trace)
COL = {"prompt": HIVE[1], "accepted": HIVE[3], "resample": HIVE[2], "bonus": HIVE[5]}
pieces = [(c, "prompt", -1) for c in decode(prompt)]
for r, ev in enumerate(trace):
    pieces += [(c, "accepted", r) for c in decode(ev["accepted"])]
    pieces.append((decode([ev["new"]]), ev["kind"], r))

WIDTH = 62
cells, row, col = [], 0, 0
for ch, kind, r in pieces:              # lay characters out on a grid, honouring newlines
    if ch == "\n":
        row, col = row + 1, 0
        continue
    if col >= WIDTH:
        row, col = row + 1, 0
    cells.append((row, col, ch, kind, r)); col += 1
n_rows, rounds = row + 1, len(trace)
frame_rounds = sorted(set(np.linspace(0, rounds - 1, min(rounds, 45)).astype(int).tolist()))

fig, ax = plt.subplots(figsize=(12, 0.3 * n_rows + 1.3))
def draw(fi):
    r_max = frame_rounds[fi]
    ax.clear(); ax.axis("off"); ax.set_xlim(-1, WIDTH + 1); ax.set_ylim(n_rows, -1)
    for rr, cc, ch, kind, r in cells:
        if r <= r_max:
            ax.text(cc, rr, ch, family="monospace", fontsize=10, color=COL[kind], ha="center", va="center",
                    fontweight="normal" if kind in ("prompt", "accepted") else "bold")
    n_tok = sum(len(ev["accepted"]) + 1 for ev in trace[:r_max + 1])
    ax.set_title(f"target calls: {r_max + 1}  ·  tokens generated: {n_tok}  ·  "
                 "teal = accepted draft, coral = resampled, purple = bonus", fontsize=10)
anim = animation.FuncAnimation(fig, draw, frames=len(frame_rounds), interval=350)
plt.close(fig)
HTML(anim.to_jshtml())

### 🔮 Predict-then-run
As γ (draft length) grows, what happens to (a) the acceptance rate per drafted token, (b) the number of tokens per target call, and (c) wall-clock speed? Is there a sweet spot?

In [ ]:
N_SPEC = 60 if FAST else 200
gammas = [1, 2, 4] if FAST else [1, 2, 3, 4, 6, 8]
plain_generate(target, prompt, 8)                                   # warm-up
t0 = time.perf_counter(); plain_generate(target, prompt, N_SPEC); t_plain = time.perf_counter() - t0
rows = []
for gm in gammas:
    t0 = time.perf_counter(); _, st = speculative_generate(target, draft, prompt, N_SPEC, gamma=gm); dt = time.perf_counter() - t0
    a = st["accepted"] / st["evaluated"]          # per-draft acceptance rate (alpha)
    rows.append(dict(gamma=gm, alpha=a, tok_per_call=N_SPEC / st["target_calls"], speedup=t_plain / dt))
    print(f"γ={gm}: acceptance {a:.2f} | tokens/target-call {N_SPEC / st['target_calls']:.2f} "
          f"(theory {(1 - a ** (gm + 1)) / (1 - a + 1e-9):.2f}) | wall-clock speedup x{t_plain / dt:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(gammas, [r["tok_per_call"] for r in rows], "o-", label="measured")
axes[0].plot(gammas, [(1 - r["alpha"] ** (r["gamma"] + 1)) / (1 - r["alpha"] + 1e-9) for r in rows], "s--", label="theory")
axes[0].set_xlabel("γ (draft tokens per round)"); axes[0].set_ylabel("tokens per target forward"); axes[0].legend()
axes[1].plot(gammas, [r["speedup"] for r in rows], "o-", color=HIVE[2]); axes[1].axhline(1, ls=":", color="gray")
axes[1].set_xlabel("γ"); axes[1].set_ylabel("speedup vs plain decoding"); axes[1].set_title(f"wall-clock ({DEVICE})")
plt.tight_layout(); plt.show()

**Reading the plot honestly.** Tokens-per-target-call rises with γ and closely tracks the i.i.d. formula $\frac{1-\alpha^{\gamma+1}}{1-\alpha}$, with diminishing returns as γ grows (each extra draft only pays off if *all* earlier ones were accepted). Wall-clock speedup depends on the **cost ratio** between draft and target: here the "big" model is tiny, so on CPU both models' forward passes cost about the same fixed overhead, and the draft's γ sequential calls eat the savings. Expect a speedup around or even below 1× here. With a large target and a small draft on a GPU, where the target pass is dominated by streaming tens of GB of weights, the original papers reported roughly 2–3× speedups. Production variants replace the separate draft with extra prediction heads (Medusa), feature-level drafting (EAGLE), or n-gram lookup from the prompt; the verification rule above stays the same.

> **🍯 Key takeaway:** Accept with min(1, p/q), otherwise resample from max(0, p − q). The draft model changes only the *speed*, never the distribution you sample from.

## Part 5 · Weight quantization: int8 & int4 🗜️

Decoding streams every weight through memory per token, so **fewer bytes per weight ≈ proportionally faster decoding** (and models that fit on smaller GPUs). Quantization maps float weights to small integers plus a scale.

**Symmetric absmax quantization** to $b$ bits, with $q_{max} = 2^{b-1}-1$ (127 for int8, 7 for int4):
$$s = \frac{\max |W|}{q_{max}},\qquad W_q = \operatorname{clamp}\!\big(\operatorname{round}(W/s), -q_{max}, q_{max}\big),\qquad \hat W = s\,W_q.$$

**Granularity is everything.** A single outlier inflates $s$ and crushes every other weight into a few levels. Fix: compute the scale over smaller groups:
* **per-tensor**: one scale for the whole matrix (cheapest, most fragile);
* **per-channel**: one scale per output row;
* **group-wise**: one scale per contiguous group of $g$ input weights within a row (e.g. $g = 32, 64, 128$). This is the standard for 4-bit LLM weights (GPTQ, AWQ, llama.cpp's Q4 formats). Cost: one fp16 scale per group, i.e. $16/g$ extra bits per weight.

(Asymmetric "zero-point" schemes add an offset so the integer grid covers $[\min W, \max W]$; GPTQ/AWQ add smarter rounding or scaling on top. Here we stick to plain round-to-nearest, the baseline they improve on.)

In [ ]:
def quantize_absmax(w, bits=8, dim=None):
    """Symmetric round-to-nearest. dim=None -> one scale; else scale per slice along `dim` (keepdim)."""
    qmax = 2 ** (bits - 1) - 1
    amax = w.abs().amax() if dim is None else w.abs().amax(dim=dim, keepdim=True)
    scale = amax.clamp(min=1e-12) / qmax
    q = torch.clamp(torch.round(w / scale), -qmax, qmax).to(torch.int8)
    return q, scale

def dequantize(q, scale):
    return q.float() * scale

### 🧪 Your Turn #5 — group-wise quantization
For `w` of shape `(out, in)` with `in % group_size == 0`: reshape to `(out, in // group_size, group_size)`, quantize with `quantize_absmax(..., dim=-1)` so each group gets its own scale, and return `(q, scale)`. Then write `dequantize_groupwise(q, scale, shape)` that undoes the reshape.

In [ ]:
def quantize_groupwise(w, bits=4, group_size=64):
    # TODO
    raise NotImplementedError

def dequantize_groupwise(q, scale, shape):
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def quantize_groupwise(w, bits=4, group_size=64):
    out_f, in_f = w.shape
    assert in_f % group_size == 0, "in_features must be divisible by group_size"
    return quantize_absmax(w.reshape(out_f, in_f // group_size, group_size), bits=bits, dim=-1)

def dequantize_groupwise(q, scale, shape):
    return (q.float() * scale).reshape(shape)

In [ ]:
def fake_quant(w, scheme):
    """Quantize then dequantize a 2-D weight with a named scheme -> float tensor of same shape."""
    kind, bits = scheme["kind"], scheme["bits"]
    if kind == "tensor":  return dequantize(*quantize_absmax(w, bits))
    if kind == "channel": return dequantize(*quantize_absmax(w, bits, dim=1))
    if kind == "group":   return dequantize_groupwise(*quantize_groupwise(w, bits, scheme["g"]), w.shape)
    raise ValueError(kind)

def rel_err(w, w_hat):
    return ((w - w_hat).norm() / w.norm()).item()

# ✔️ Checkpoint
torch.manual_seed(0)
W = torch.randn(64, 256) * 0.02
q, s = quantize_groupwise(W, 4, 64)
assert q.shape == (64, 4, 64) and s.shape == (64, 4, 1) and q.dtype == torch.int8
assert q.abs().max() <= 7
W_hat = dequantize_groupwise(q, s, W.shape)
assert W_hat.shape == W.shape
assert (W - W_hat).abs().max() <= s.max() / 2 + 1e-7, "error must be at most half a quantization step"
assert rel_err(W, fake_quant(W, {"kind": "group", "bits": 4, "g": 64})) < rel_err(W, fake_quant(W, {"kind": "tensor", "bits": 4}))
print("✔️ group-wise quantization works")

In [ ]:
# Outliers: the reason granularity matters. Plant ONE large weight and compare schemes.
schemes = {"int8 tensor": {"kind": "tensor", "bits": 8}, "int8 channel": {"kind": "channel", "bits": 8},
           "int4 tensor": {"kind": "tensor", "bits": 4}, "int4 channel": {"kind": "channel", "bits": 4},
           "int4 g64": {"kind": "group", "bits": 4, "g": 64}, "int4 g32": {"kind": "group", "bits": 4, "g": 32}}
W = target.blocks[1].mlp.w_up.weight.detach().cpu().clone()
W_out = W.clone(); W_out[3, 5] = 25 * W.abs().max()
clean = [rel_err(W, fake_quant(W, s)) for s in schemes.values()]
dirty = [rel_err(W_out, fake_quant(W_out, s)) for s in schemes.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
x = np.arange(len(schemes))
axes[0].bar(x - 0.2, clean, 0.4, label="real weights"); axes[0].bar(x + 0.2, dirty, 0.4, label="+ one outlier", color=HIVE[2])
axes[0].set_xticks(x); axes[0].set_xticklabels(schemes.keys(), rotation=20); axes[0].set_yscale("log")
axes[0].set_ylabel("relative error ‖W−Ŵ‖/‖W‖"); axes[0].legend(); axes[0].set_title("one outlier wrecks per-tensor scales")
row = W[0, :64]
qt = fake_quant(W, schemes["int4 tensor"])[0, :64]; qg = fake_quant(W, schemes["int4 g64"])[0, :64]
axes[1].plot(row.numpy(), "o", ms=4, label="original", color=HIVE[1])
axes[1].plot(qt.numpy(), "_", ms=10, mew=2, label="int4 per-tensor", color=HIVE[2])
axes[1].plot(qg.numpy(), "_", ms=10, mew=2, label="int4 group-64", color=HIVE[3])
axes[1].set_title("first 64 weights of one row: tensor-wide scale leaves only a few levels"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
def quantize_model(model, scheme):
    qm = copy.deepcopy(model)
    with torch.no_grad():
        for mod in qm.modules():
            if isinstance(mod, nn.Linear):          # includes lm_head (tied to the embedding)
                mod.weight.copy_(fake_quant(mod.weight.float().cpu(), scheme).to(mod.weight.device, mod.weight.dtype))
    return qm

def bits_per_weight(scheme):
    if scheme is None: return 32
    extra = 16 / scheme["g"] if scheme["kind"] == "group" else 0      # one fp16 scale per group
    return scheme["bits"] + extra

eval_schemes = {"fp32": None, "int8 tensor": schemes["int8 tensor"], "int8 channel": schemes["int8 channel"],
                "int4 tensor": schemes["int4 tensor"], "int4 channel": schemes["int4 channel"],
                "int4 g32": schemes["int4 g32"], "int3 g32": {"kind": "group", "bits": 3, "g": 32},
                "int2 g32": {"kind": "group", "bits": 2, "g": 32}}
results = {}
for name, sch in eval_schemes.items():
    m = target if sch is None else quantize_model(target, sch)
    results[name] = eval_loss(m)
    print(f"{name:13s} | ~{bits_per_weight(sch):5.2f} bits/weight | val loss {results[name]:.4f} | perplexity {math.exp(results[name]):.3f}")

fig, ax = plt.subplots(figsize=(10, 3.8))
names = list(results)
ax.bar(names, [math.exp(results[k]) for k in names], color=[HIVE[3] if k == "fp32" else HIVE[0] for k in names], edgecolor=HIVE[1])
ax.axhline(math.exp(results["fp32"]), ls="--", color=HIVE[1], lw=1)
ax.set_ylabel("val perplexity (char-level)"); ax.set_title("perplexity after round-to-nearest weight quantization")
plt.xticks(rotation=20); plt.show()

**What you should see:** int8 is essentially free; int4 *per-tensor* hurts noticeably; int4 with **small groups** recovers most of the gap at ~4.5 bits/weight; int3 starts to bend and int2 breaks (that's where GPTQ/AWQ-style error compensation or quantization-aware training become necessary). Tiny models are *more* fragile than big ones per bit, so these numbers are pessimistic for real LLMs.

**Memory math for an 8B-parameter model:** fp16 ≈ 16 GB → int8 ≈ 8 GB → int4 g128 ≈ 4.1 bits/weight ≈ 4.1 GB (plus KV cache and activations). That's the difference between needing a large-memory GPU and running comfortably on a laptop.

### ⚡ Optional: the same experiment on a real pretrained LM (distilgpt2)
Downloads ~350 MB from the Hugging Face Hub (skipped in FAST mode or when offline). GPT-2's `Conv1D` layers store weights as `(in, out)`, so we transpose before group-quantizing along the input dimension.

In [ ]:
RUN_HF = (not FAST) or os.environ.get("LAB_HF", "0") == "1"
if RUN_HF:
    try:
        warnings.filterwarnings("ignore")                           # silence tqdm / hub chatter
        os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
        from transformers import AutoTokenizer, AutoModelForCausalLM
        from transformers.utils import logging as hf_logging
        hf_logging.set_verbosity_error()
        hf_tok = AutoTokenizer.from_pretrained("distilgpt2")
        hf_model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(DEVICE).eval()
        passage = val_data[:6000]
        enc_ids = hf_tok(decode(passage.tolist()), return_tensors="pt", truncation=True, max_length=1024).input_ids.to(DEVICE)

        @torch.no_grad()
        def hf_ppl(m):
            return math.exp(m(enc_ids, labels=enc_ids).loss.item())

        def hf_quantize(model, scheme):
            qm = copy.deepcopy(model)
            with torch.no_grad():
                for mod in qm.modules():
                    if type(mod).__name__ == "Conv1D":                 # weight is (in, out)
                        w = mod.weight.data.float().cpu().T             # -> (out, in)
                        mod.weight.data.copy_(fake_quant(w, scheme).T.to(mod.weight.device))
            return qm

        hf_res = {"fp32": hf_ppl(hf_model)}
        for name in ["int8 channel", "int4 tensor", "int4 channel", "int4 g32"]:
            hf_res[name] = hf_ppl(hf_quantize(hf_model, schemes[name]))
        for k, v in hf_res.items():
            print(f"distilgpt2 {k:13s}: perplexity on a Shakespeare passage = {v:8.2f}")
    except Exception as e:
        print("Skipping the Hugging Face section:", repr(e))
else:
    print("FAST mode: skipping the distilgpt2 download.")

> **🍯 Key takeaway:** Quantization error is set by the scale, and the scale by outliers. Smaller groups give much better int4 for ~16/g extra bits per weight; int8 is nearly free.

## Part 6 · Concepts: FlashAttention & PagedAttention 🧠

### FlashAttention: IO-awareness
Standard attention writes the $T \times T$ score matrix to GPU main memory (HBM), reads it back for softmax, writes the probabilities, reads them again for $PV$. At $T = 8{,}192$ that's 67M entries *per head* bouncing through HBM, and HBM bandwidth (not FLOPs) becomes the bottleneck.

**FlashAttention** (Dao et al., 2022) never materialises the full matrix. It tiles $Q, K, V$ into blocks that fit in on-chip SRAM and computes softmax **online**: for each query block it streams over key blocks while maintaining a running max $m$, running normaliser $\ell$ and running output $o$:
$$m' = \max(m, \max_j s_j),\quad \ell' = e^{m-m'}\ell + \sum_j e^{s_j - m'},\quad o' = e^{m-m'} o + \sum_j e^{s_j-m'} v_j,$$
and divides by $\ell$ at the end. The result is **exact** (not an approximation), uses $O(T)$ extra memory instead of $O(T^2)$, and is several times faster because it is IO-bound work done in SRAM. The backward pass recomputes attention on the fly rather than storing it. FlashAttention-2 improves work partitioning and parallelism, and FlashAttention-3 exploits Hopper-GPU features; `F.scaled_dot_product_attention` dispatches to such kernels on supported GPUs.

Let's verify the online-softmax trick in plain PyTorch (this is slow in Python; the point is the math, not the speed):

In [ ]:
def tiled_attention(q, k, v, block=16):
    """Exact causal attention computed block-by-block over keys with an online softmax (FlashAttention's core idea)."""
    T, d = q.shape
    out = torch.zeros_like(q)
    m = torch.full((T, 1), float("-inf")); l = torch.zeros(T, 1)
    for j0 in range(0, T, block):
        kj, vj = k[j0:j0 + block], v[j0:j0 + block]
        s = q @ kj.T / math.sqrt(d)                                       # (T, block) — only one tile at a time
        qi = torch.arange(T).unsqueeze(1); kj_idx = torch.arange(j0, j0 + kj.shape[0]).unsqueeze(0)
        s = s.masked_fill(kj_idx > qi, float("-inf"))                     # causal
        m_new = torch.maximum(m, s.amax(-1, keepdim=True))
        corr = torch.exp(m - m_new).nan_to_num(0.0)                       # rescale old stats
        p = torch.exp(s - m_new).nan_to_num(0.0)
        l = corr * l + p.sum(-1, keepdim=True)
        out = corr * out + p @ vj
        m = m_new
    return out / l

torch.manual_seed(0)
q_, k_, v_ = (torch.randn(100, 32) for _ in range(3))
ref = F.scaled_dot_product_attention(q_[None], k_[None], v_[None], is_causal=True)[0]
assert torch.allclose(tiled_attention(q_, k_, v_, block=16), ref, atol=1e-5)
print("✔️ tiled online-softmax attention is exact")

Ts = np.array([1024, 2048, 4096, 8192, 16384, 32768])
naive_mb = Ts ** 2 * 4 / 2**20                    # one fp32 score matrix, one head
flash_mb = Ts * (2 * 4) / 2**20                   # running max + normaliser per query (fp32)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.loglog(Ts, naive_mb, "o-", label="standard: T×T scores in HBM"); ax.loglog(Ts, flash_mb, "s-", label="FlashAttention: O(T) statistics")
ax.set_xlabel("sequence length T"); ax.set_ylabel("extra memory per head (MiB)"); ax.legend(); ax.set_title("attention intermediate memory")
plt.show()

### PagedAttention: virtual memory for the KV cache
A server handles many requests of unknown final length. Pre-allocating a contiguous `max_len` KV buffer per request (as our `KVCache` does!) wastes most of it, and fragmentation limits how many requests fit in GPU memory at once.

**PagedAttention** (Kwon et al., 2023, the core of vLLM) splits each request's KV cache into fixed-size **blocks** (e.g. 16 tokens), allocated on demand from a shared pool and tracked by a per-request **block table**, exactly like OS virtual-memory pages. Waste is limited to the last, partially-filled block; blocks can be **shared** between requests with a common prefix (system prompts, parallel samples, beam search) with copy-on-write. Combined with **continuous batching** (adding/removing requests from the running batch at every step), the vLLM paper reports 2–4× higher throughput than prior serving systems at similar latency. Let's simulate the memory waste:

In [ ]:
rng = np.random.default_rng(0)
MAX_LEN, BLOCK = 2048, 16
lengths_req = np.clip(rng.lognormal(mean=5.5, sigma=0.9, size=200).astype(int), 8, MAX_LEN)   # actual tokens used
contig_reserved = MAX_LEN * len(lengths_req)
paged_reserved = int(np.sum(np.ceil(lengths_req / BLOCK) * BLOCK))
used = int(lengths_req.sum())
print(f"tokens actually used      : {used:,}")
print(f"contiguous (max_len) alloc: {contig_reserved:,}  -> {100 * (1 - used / contig_reserved):.1f}% wasted")
print(f"paged ({BLOCK}-token blocks)   : {paged_reserved:,}  -> {100 * (1 - used / paged_reserved):.1f}% wasted")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(lengths_req, bins=40, color=HIVE[0], edgecolor=HIVE[1]); axes[0].set_xlabel("request length (tokens)"); axes[0].set_title("simulated request lengths")
axes[1].bar(["used", "paged", "contiguous"], [used, paged_reserved, contig_reserved], color=[HIVE[3], HIVE[0], HIVE[2]])
axes[1].set_ylabel("KV slots reserved"); axes[1].set_yscale("log"); axes[1].set_title("reserved KV-cache slots for 200 requests")
plt.tight_layout(); plt.show()

> **🍯 Key takeaway:** FlashAttention is *exact* attention reorganised around memory traffic (tiles + online softmax); PagedAttention is virtual memory for the KV cache.

## 🏆 Boss Level
1. **Batched KV-cache generation with ragged prompts.** Extend `KVCache`/`generate_cached` to a batch of prompts of different lengths (left-padding + an attention mask, or per-sequence `start_pos`). Measure throughput vs batch size 1, 4, 16, 64 and find where decoding stops being "free" to batch.
2. **KV-cache quantization.** Store K and V in int8 with a per-token (or per-head) absmax scale, dequantize on read, and measure the perplexity hit and the memory saving. Which is more sensitive, K or V?
3. **Prompt-lookup / n-gram speculation.** Replace the draft model with a "draft" that copies the continuation of the most recent earlier occurrence of the last 3 tokens in the context (zero extra parameters). Implement it with the *greedy* acceptance variant (accept while the draft equals the target's argmax), and measure tokens per target call on Shakespeare passages with repeated names.

## 🍯 Key Takeaways
- **KV cache**: store past keys/values so each decode step processes one token. Same outputs, far less compute, but memory of $2 \cdot L \cdot n_{kv} \cdot d_{head}$ numbers per token, which is why GQA, MLA, KV quantization and paging exist.
- **Decoding is memory-bound**: every step streams all the weights to produce one token per sequence; batching, speculation and quantization all attack that.
- **Sampling**: temperature reshapes; top-k truncates rigidly; **top-p** and **min-p** adapt to the model's confidence and keep high-temperature text coherent.
- **Speculative decoding** is *lossless*: accept with $\min(1, p/q)$, resample from $\max(0, p-q)$; the acceptance rate is $1-\mathrm{TV}(p,q)$.
- **Quantization**: granularity tames outliers. int8 is nearly free, int4 with small groups costs a little, and 2–3 bits need smarter methods (GPTQ, AWQ, QAT).
- **FlashAttention** = exact attention computed in tiles with an online softmax; **PagedAttention** = KV cache in blocks with block tables (near-zero waste, prefix sharing).

## 🎤 Explain it to a friend
Your friend asks why a chatbot sometimes "types" slowly and what engineers do about it. In one paragraph, explain (1) why it remembers its own past work instead of re-reading everything (the KV cache), (2) how an "intern" model can draft words that the "senior" model quickly checks without ever changing what the senior would have said (speculative decoding), and (3) why storing each number in 4 bits instead of 16 makes it both smaller and faster.

## 💼 Interview corner
1. **Why is LLM decoding at batch size 1 memory-bandwidth bound, and how does batching help?**
   <details><summary>Answer</summary>Each decode step must read every weight (and the KV cache) from GPU memory to do only about 2 FLOPs per weight per sequence, an arithmetic intensity far below what the hardware can sustain, so the step time is set by bytes moved. Batching B sequences reuses each weight read for B tokens, raising arithmetic intensity until compute becomes the limit.</details>
2. **Why does speculative sampling produce exactly the target distribution? What controls the speedup?**
   <details><summary>Answer</summary>A drafted x ~ q is kept with probability min(1, p(x)/q(x)), contributing min(p, q); on rejection we sample from the normalised residual max(0, p − q), which contributes exactly max(0, p − q). The sum is p. The speedup depends on the acceptance rate α = Σ min(p, q), the draft length γ (expected tokens per target call (1 − α^{γ+1})/(1 − α)), and the cost ratio of draft to target forward passes.</details>
3. **Why does group-wise int4 beat per-tensor int4, and what does it cost?**
   <details><summary>Answer</summary>Absmax scaling sets the step size from the largest |w| in the group, so one outlier makes the step huge and rounds most weights to a few levels. Smaller groups confine outliers' damage to their own group. The cost is one scale per group: with fp16 scales and group size g, that's 16/g extra bits per weight (0.5 bits at g = 32), plus a little dequantization work.</details>

## 🐝 Fun fact
Speculative decoding was invented twice, independently and almost simultaneously: Leviathan, Kalman & Matias at Google (arXiv, Nov 2022) and Chen et al. at DeepMind (arXiv, Feb 2023) arrived at the same accept/reject rule.

## 📚 Further reading
- Pope et al., *Efficiently Scaling Transformer Inference* (2022).
- Holtzman et al., *The Curious Case of Neural Text Degeneration* (ICLR 2020) — nucleus sampling.
- Nguyen et al., *Turning Up the Heat: Min-p Sampling for Creative and Coherent LLM Outputs* (2024).
- Leviathan, Kalman & Matias, *Fast Inference from Transformers via Speculative Decoding* (ICML 2023).
- Chen et al., *Accelerating Large Language Model Decoding with Speculative Sampling* (2023).
- Cai et al., *Medusa: Simple LLM Inference Acceleration Framework with Multiple Decoding Heads* (2024); Li et al., *EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty* (2024).
- Dettmers et al., *LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale* (2022).
- Frantar et al., *GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers* (ICLR 2023).
- Lin et al., *AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration* (MLSys 2024).
- Dao et al., *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness* (2022); Dao, *FlashAttention-2* (2023).
- Milakov & Gimelshein, *Online normalizer calculation for softmax* (2018).
- Kwon et al., *Efficient Memory Management for Large Language Model Serving with PagedAttention* (SOSP 2023).
- Shazeer, *Fast Transformer Decoding: One Write-Head is All You Need* (2019) — multi-query attention.